In [33]:

import glob
import pandas as pd
import os
import pyarrow as pa
import numpy as np
import pyarrow.parquet as pq
import re


from scipy.stats import spearmanr, pearsonr


In [34]:
from load_experiment_data import (
    train_dataset_name,
    test_dataset_name,
    train_dataset_split,
    test_dataset_split,
    load_data_and_estimators,
    explanation_types,
    explanation_m,
    linear_coders,
    explanation_k,
    explanation_lambda,
    explanation_seed
    )

In [35]:
NUM_TEST_INSTANCES = 1000 

In [36]:
k = len(explanation_k)
m = 1 # len(explanation_m)
lambda_ = len(explanation_lambda)
explanation_seed_ = 5

In [37]:
num_explanation_types = 1 # Self

num_explanation_types += k*m # explanations.TopKMostInfluential,
num_explanation_types += k*m # explanations.TopKLeastInfluential,
num_explanation_types += k*m # explanations.TopKMostHelpful,
num_explanation_types += k*m # explanations.TopKMostHarmful,
num_explanation_types += k*m*lambda_ # explanations.FacilityLocationMostHarmful,
num_explanation_types += k*m*lambda_ # explanations.FacilityLocationMostHelpful,
num_explanation_types += k*m*lambda_ # explanations.FacilityLocationMostInfluential,
num_explanation_types += k*m*lambda_ # explanations.FacilityLocationLeastInfluential,
num_explanation_types += k*m # explanations.DIVINEMostHelpful,
num_explanation_types += k*m # explanations.DIVINEMostHarmful,
num_explanation_types += k*m # explanations.DIVINEMostInfluential,
num_explanation_types += k*m # explanations.DIVINELeastInfluential,
num_explanation_types += k*m # explanations.AIDE
num_explanation_types += k*m*explanation_seed_
num_explanation_types

137

In [38]:
num_explanation_types_validation = num_explanation_types
num_explanation_types_validation -= 1 # AIDE

In [39]:
df_validation = pq.ParquetDataset("results/validation").read().to_pandas()


In [40]:
len(df_validation)

448294

In [41]:
df_scoring = pq.ParquetDataset("results/scoring").read().to_pandas()

In [42]:
group_sizes = df_scoring.groupby(
    ["model","explanation_type","train_dataset","train_split","test_dataset","test_split",
     "estimator", "linear_coder"]
).size()

group_sizes[group_sizes != NUM_TEST_INSTANCES]

model                                                        explanation_type                                                                                         train_dataset                                 train_split  test_dataset                                  test_split  estimator                                   linear_coder                     
Qwen2.5-0.5B_tulu-3-sft-olmo-2-mixture-0225_lr0.0001_seed42  25 by facility location from Top-100 most harmful (most positive scores). lambda=0.0                     loris3/tulu-3-sft-olmo-2-mixture-0225-sample  train        loris3/tulu-3-sft-olmo-2-mixture-0225-sample  test        LESSEstimator: normalize=True               KLTCoder                             749
                                                                                                                                                                                                                                                                               

In [43]:
set(df_scoring["explanation_type"].unique()) - set(df_validation["explanation_type"].unique())

set()

In [44]:
len(df_validation["explanation_type"].unique()) == num_explanation_types_validation

False

In [45]:
group_sizes = df_validation.groupby(
    ["model","explanation_type", "train_dataset","train_split","test_dataset","test_split",
     "estimator"]
).size()

group_sizes[group_sizes != NUM_TEST_INSTANCES]

model                                                          explanation_type                                                                       train_dataset                                 train_split  test_dataset                                  test_split  estimator                                 
OLMo-2-0425-1B_tulu-3-sft-olmo-2-mixture-0225_lr0.0001_seed42  5 by AIDE from Top-100.                                                                loris3/tulu-3-sft-olmo-2-mixture-0225-sample  train        loris3/tulu-3-sft-olmo-2-mixture-0225-sample  test        DataInfEstimator: fast_implementation=True    945
Qwen2.5-0.5B_tulu-3-sft-olmo-2-mixture-0225_lr0.0001_seed42    1 by facility location from Top-100 most helpful (most negative scores). lambda=0.0    loris3/tulu-3-sft-olmo-2-mixture-0225-sample  train        loris3/tulu-3-sft-olmo-2-mixture-0225-sample  test        DataInfEstimator: fast_implementation=True    504
                                                        

In [46]:
len(df_validation["explanation_type"].unique()) 

137

In [47]:
len(df_scoring["explanation_type"].unique()) == num_explanation_types

True

In [48]:
len(df_scoring)

4148700

In [49]:
group_sizes = df_validation.groupby(
    ["train_dataset","train_split","test_dataset","test_split",
     "model","estimator","explanation_type"]
).size()

group_sizes[group_sizes != NUM_TEST_INSTANCES]

train_dataset                                 train_split  test_dataset                                  test_split  model                                                          estimator                                   explanation_type                                                                     
loris3/tulu-3-sft-olmo-2-mixture-0225-sample  train        loris3/tulu-3-sft-olmo-2-mixture-0225-sample  test        OLMo-2-0425-1B_tulu-3-sft-olmo-2-mixture-0225_lr0.0001_seed42  DataInfEstimator: fast_implementation=True  5 by AIDE from Top-100.                                                                  945
                                                                                                                     Qwen2.5-0.5B_tulu-3-sft-olmo-2-mixture-0225_lr0.0001_seed42    BM25Estimator: k1=1.5, b=0.75               10 random examples with seed 9                                                           380
                                                        